In [1]:
import importlib
import os
import pickle
from pathlib import Path
import re
import datetime

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

from behave_analysis.visualize.visualize_utils import open_tracking_data
from behave_analysis.process.session import get_experiment
from behave_analysis.utils.creating_directories import make_directory
from behave_analysis.analyze.single_trial import preprocess_regression
from behave_analysis.analyze.single_trial import single_trial_regression
from behave_analysis.analyze.single_trial.single_trial_regression import SingleTrialRegression
from behave_analysis.analyze.single_trial.preprocess_regression import PreprocessSingleTrialRegression
from behave_analysis.utils.label_barrier_edges import check_which_barrier_location_is_which_orientation, convert_left_right_to_pre_post_flip

## Import data

In [2]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept
from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept
from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept
from behave_analysis.database.Experiments.JAL006_ex import JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr
from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr, JAL7_30apr
from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_tiny_3may, JAL8_flip4_10may, JAL8_14may, JAL8_21may

# Sessions to consider

In [3]:
tinny_barrier = [JAL8_tiny_3may, JAL8_21may, JAL7_30apr]

# Hash mice names to experiment names
mice_groups = {
    "JAL3": ['JAL3_25aug', 'JAL3_1sept', 'JAL3_4sept', 'JAL3_7sept'],
    "JAL4": ['JAL4_3rdSept', 'JAL4_19thSept', 'JAL4_28aug', 'JAL4_11thSept'],
    "JAL5": ['JAL5_8thSept', 'JAL5_21stSept'],
    "JAL6": ['JAL6_flip7_1apr', 'JAL6_flip3_18mar', 'JAL6_flip4_21mar', 'JAL6_flip5_25mar', 'JAL6_28mar'],
    "JAL7": ['JAL7_sesh8_9apr', 'JAL7_sesh9_16apr', 'JAL7_flip5_22mar', 'JAL7_flip2_12mar', 'JAL7_23apr'],
    "JAL8": ['JAL8_flip1_25apr', 'JAL8_flip2_29apr', 'JAL8_flip4_10may', 'JAL8_14may']}

# Hash mice names to experiment objects
mice_to_experiment = {
    "JAL3": [JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept],
    "JAL4": [JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept],
    "JAL5": [JAL005_8thSept, JAL005_21stSept],
    "JAL6": [JAL6_flip7_1apr, JAL6_flip3_18mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_28mar],
    "JAL7": [JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr],
    "JAL8": [JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may]}


# Upload a test synthetic file

In [8]:
mice_to_experiment = {"JAL7": [JAL7_sesh9_16apr, JAL7_sesh8_9apr]}
mice_groups = {"JAL7": ['JAL7_sesh9_16apr', "JAL7_sesh8_9apr"]}

# Init paths and settings

In [9]:
c_type = "synthetic" # good or synthetic
dir = make_directory(r"Z:\Jasmine_Laurence\single_trial_overview")
dir = Path(dir)
conditions = ["shelter_only", "barrier_pre_flip", "barrier_post_flip"]

# Loop through sessions and generate R2 scores across all homings all conditions

In [10]:
for mouse, experiments in mice_to_experiment.items():
    print(experiments)
    for i, session in enumerate(experiments):

        print(f"Running single trial analysis for {mouse} on session {mice_groups[mouse][i]}")
        loaded_session = get_experiment(session)

        # Use the experiment object to load the data
        try:
            video_df = pl.read_csv(os.path.join(loaded_session.base_path, loaded_session.processed_path) + "\\" "full_video_dataframe.csv")
            homing_path = os.path.join(loaded_session.base_path, loaded_session.processed_path, "homings", "homings_obj.pkl")
            with open(homing_path, "rb") as hf:
                    homings_object = pickle.load(hf) 
            video_and_spike_data_path = os.path.join(loaded_session.base_path, loaded_session.processed_path, c_type +"_video_spike_count_df.parquet")
            video_and_spike_data = pl.read_parquet(video_and_spike_data_path)
            frame_by_cluster_matrix = np.load(os.path.join(loaded_session.base_path, loaded_session.processed_path) + "\\" + "frame_by_" + c_type + "_cluster_matrix.npy")
            tracking_data = open_tracking_data(loaded_session)
            cluster_Ids = np.load(str(os.path.join(loaded_session.base_path, loaded_session.processed_path) + "/" + c_type + "_cluster_Ids.npy"))
        
        except FileNotFoundError:
            print("One of the files was not found")
        
        # If there is no barrier in this session hard code it based on a prior session
        if not "barrier_loc" in tracking_data.keys():
            tracking_data["barrier_loc"] = [[224, 515], [797, 512], [510, 513]]
        
        # For each condition run the single trial analysis - This will be saved as a separate file per condition, mouse and session
        for condition in conditions:
            pp_single_trial_obj = PreprocessSingleTrialRegression(
                video_df=video_df,
                homings_obj=homings_object,
                frame_by_cluster_matrix=frame_by_cluster_matrix,
                save_path=dir,
                velocity_data=tracking_data["avg_Velocity"],
                barrier_location=tracking_data["barrier_loc"],
                shelter_location=tracking_data["shelter_loc"],
                save_plots=False,
                condition=condition)
            
            if len(pp_single_trial_obj.design_matrix) == 0:
                print("No trials for this condition")
                continue
            
            # Make sure there are enough homings in this condition 
            if pp_single_trial_obj.condition_per_homing.count(condition) < 10:
                print("Less than 10 trials for this condition")
                continue
                    
            SingleTrialRegression(
                session=loaded_session,
                design_matrix=pp_single_trial_obj.design_matrix,
                save_path=dir,
                dependents_df=pp_single_trial_obj.targets_df,
                tracking_data=tracking_data,
                homing_list=pp_single_trial_obj.homing_list,
                spike_homing_list=pp_single_trial_obj.spike_data_per_homing,
                condition_per_homing=pp_single_trial_obj.condition_per_homing,
                cluster_ids=cluster_Ids,
                initial_directions=None,
                conversion_from_left_right_to_pre_post_flip=convert_left_right_to_pre_post_flip(tracking_data["barrier_loc"]),
                session_name=mice_groups[mouse][i],
                condition=condition)                    

[Experiment(nick_name='JAL007', total_sessions=9, mouse_number_pyrat='BAA-1104293', experiment_file_names=None, root_path=WindowsPath('JAL007'), experiment_name='sesh8', experiment_idx=0, experiment_date='2024_04_16', experiment_time='11_13_05', experiment_path=WindowsPath('JAL007_shelter_barrier_flip_9_2024_04_16T11_13_05'), shelter_time=[3.75, -1], barrier_time=[73.5, -1], barrier_flip_time=200), Experiment(nick_name='JAL007', total_sessions=9, mouse_number_pyrat='BAA-1104293', experiment_file_names=None, root_path=WindowsPath('JAL007'), experiment_name='sesh8', experiment_idx=0, experiment_date='2024_04_09', experiment_time='10_07_45', experiment_path=WindowsPath('JAL007_shelter_barrier_flip_8_2024_04_09T10_07_45'), shelter_time=[0, -1], barrier_time=[116.5, -1], barrier_flip_time=227)]
Running single trial analysis for JAL7 on session JAL7_sesh9_16apr


2024-11-04 16:17:11.488 | SUCCESS  | behave_analysis.analyze.single_trial.preprocess_regression:__init__:69 - The single trial regression preprocessing object has been initialized
2024-11-04 16:17:11.488 | INFO     | behave_analysis.utils.label_barrier_edges:check_which_barrier_location_is_which_orientation:15 - The barrier location pre flip is [796, 512] and post flip is [231, 511]
2024-11-04 16:17:11.489 | INFO     | behave_analysis.utils.label_barrier_edges:check_which_barrier_location_is_which_orientation:20 - The barrier preflip location is the right edge
2024-11-04 16:17:11.489 | INFO     | behave_analysis.analyze.single_trial.single_trial_regression:__init__:66 - Initializing the single trial regression analysis object
2024-11-04 16:17:11.524 | INFO     | behave_analysis.analyze.single_trial.single_trial_regression:run_all_dependent_variables:305 - Running the model for dependent variable: frames
2024-11-04 16:17:11.838 | INFO     | behave_analysis.analyze.single_trial.single_tr

Running single trial analysis for JAL7 on session JAL7_sesh8_9apr


2024-11-04 16:17:31.274 | SUCCESS  | behave_analysis.analyze.single_trial.preprocess_regression:__init__:69 - The single trial regression preprocessing object has been initialized
2024-11-04 16:17:31.275 | INFO     | behave_analysis.utils.label_barrier_edges:check_which_barrier_location_is_which_orientation:15 - The barrier location pre flip is [796, 512] and post flip is [231, 511]
2024-11-04 16:17:31.275 | INFO     | behave_analysis.utils.label_barrier_edges:check_which_barrier_location_is_which_orientation:20 - The barrier preflip location is the right edge
2024-11-04 16:17:31.276 | INFO     | behave_analysis.analyze.single_trial.single_trial_regression:__init__:66 - Initializing the single trial regression analysis object
2024-11-04 16:17:31.314 | INFO     | behave_analysis.analyze.single_trial.single_trial_regression:run_all_dependent_variables:305 - Running the model for dependent variable: frames


One of the files was not found


2024-11-04 16:17:31.534 | INFO     | behave_analysis.analyze.single_trial.single_trial_regression:run_all_dependent_variables:305 - Running the model for dependent variable: mouse_x_position
2024-11-04 16:17:31.766 | INFO     | behave_analysis.analyze.single_trial.single_trial_regression:run_all_dependent_variables:305 - Running the model for dependent variable: mouse_y_position
2024-11-04 16:17:31.977 | INFO     | behave_analysis.analyze.single_trial.single_trial_regression:run_all_dependent_variables:305 - Running the model for dependent variable: hdir
2024-11-04 16:17:32.193 | INFO     | behave_analysis.analyze.single_trial.single_trial_regression:run_all_dependent_variables:305 - Running the model for dependent variable: hsa
2024-11-04 16:17:32.419 | INFO     | behave_analysis.analyze.single_trial.single_trial_regression:run_all_dependent_variables:305 - Running the model for dependent variable: h_preflipbar_a
2024-11-04 16:17:32.662 | INFO     | behave_analysis.analyze.single_tria

Less than 10 trials for this condition


2024-11-04 16:17:35.073 | INFO     | behave_analysis.analyze.single_trial.single_trial_regression:run_all_dependent_variables:305 - Running the model for dependent variable: mouse_x_position
2024-11-04 16:17:35.326 | INFO     | behave_analysis.analyze.single_trial.single_trial_regression:run_all_dependent_variables:305 - Running the model for dependent variable: mouse_y_position
2024-11-04 16:17:35.572 | INFO     | behave_analysis.analyze.single_trial.single_trial_regression:run_all_dependent_variables:305 - Running the model for dependent variable: hdir
2024-11-04 16:17:35.825 | INFO     | behave_analysis.analyze.single_trial.single_trial_regression:run_all_dependent_variables:305 - Running the model for dependent variable: hsa
2024-11-04 16:17:36.073 | INFO     | behave_analysis.analyze.single_trial.single_trial_regression:run_all_dependent_variables:305 - Running the model for dependent variable: h_preflipbar_a
2024-11-04 16:17:36.322 | INFO     | behave_analysis.analyze.single_tria

# Load by condition

In [11]:
def load_r2_data(load_dir, condition):
    """
    Loads .pickle files from a directory based on a specific pattern condition.
    
    Args:
    - load_dir (str): Directory path where the .pickle files are stored.
    - condition (str): The condition pattern to match (e.g., "shelter_only", "barrier_pre_flip", "barrier_post_flip").
    
    Returns:
    - file_names (list): List of file names that match the condition.
    - r2_data (dict): Dictionary of r2 data loaded from matching files.
    """
    
    # Initialize storage for file names and r2 data
    file_names = []
    r2_data = {}

    # Create a dynamic pattern based on the condition
    pattern = fr'^(.*)_{condition}_r2'

    # Loop through files in the specified directory
    for file in os.listdir(load_dir):
        match = re.match(pattern, file)
        if match:
            extracted_part = match.group(1)
            file_names.append(extracted_part)
            
            # Load the .pickle files
            if file.endswith(".pickle"):
                with open(os.path.join(load_dir, file), "rb") as f:
                    data = pickle.load(f)
                    r2_data[extracted_part] = data
    
    return file_names, r2_data

## Across mice for each condition

In [12]:
load_dir = r"Z:\Jasmine_Laurence\single_trial_overview\r2_scores"
con_dict = {}
for condition in conditions:
    file_names, r2_data = load_r2_data(load_dir, condition)
    con_dict[condition] = pd.DataFrame(r2_data).T
con_dict["shelter_only"]

,frames,mouse_x_position,mouse_y_position,hdir,hsa,h_preflipbar_a,h_postflipbar_a,homing_id,post_flip_index,pre_flip_index,velocity,random
JAL7_sesh9_16apr,-1.253504,-0.124939,0.020482,-0.191449,0.342075,-0.695345,0.459967,-2.187501,0.096923,0.12872,-0.035122,-0.279529


# Mice names

In [157]:
mice_names = ["JAL3", "JAL4", "JAL005", "JAL6", "JAL7", "JAL8"]

## Session orders

In [158]:
JAL3_session_order = ["JAL3_25th_Aug", "JAL3_1st_Sept", "JAL3_4th_Sept", "JAL3_7th_Sept"]
JAL4_session_order = ["JAL4_28th_Aug", "JAL4_3rd_Sept","JAL4_11th_Sept", "JAL4_19th_Sept"]
JAL6_session_order = ["JAL6_4th_March", "JAL6_18th_March", "JAL6_21st_March", "JAL6_25th_March", "JAL6_28th_March"]
JAL7_session_order = ["JAL7_12th_March", "JAL7_22nd_March", "JAL7_9th_April", "JAL7_16th_April", "JAL7_23rd_April", "JAL7_30th_April"]

# For each condition, plot all variables for each mouse - Line plots

In [ ]:
def plot_mice_on_separate_axes(con_dict, conditions, mice_names):
    for condition in conditions:
        df = con_dict[condition]
        # remove columns from df
        df = df.drop(columns=["frames", "homing_id", "mouse_x_position", "random"])
        
        df[df < 0] = 0  # Clip negative values to zero

        # Create a 3x2 grid of subplots (adjust based on how many mice there are)
        fig, axs = plt.subplots(3, 2, figsize=(20, 20))
        fig.suptitle(f"R2 Scores for Condition: {condition}", fontsize=20)

        row_idx = 0
        col_idx = 0
        
        for mouse in mice_names:
            
            # Check if you need to move to the next row after filling the current column
            if col_idx == 2:
                row_idx += 1
                col_idx = 0  # Reset column index after completing a row

            # Check if we have more rows than subplot grid size, adjust accordingly
            if row_idx == 3:
                print("Warning: Too many mice for the grid size, skipping extra mice.")
                break
            
            print(f"Plotting mouse: {mouse}")
            mouse_data = df[df.index.str.contains(mouse)]
            
            if mouse == "JAL6":
                # Set the index as a Categorical type with the specified order
                mouse_data.index = pd.Categorical(mouse_data.index, categories=JAL6_session_order, ordered=True)
                mouse_data = mouse_data.sort_index()
            
            if mouse == "JAL3":
                # Set the index as a Categorical type with the specified order
                mouse_data.index = pd.Categorical(mouse_data.index, categories=JAL3_session_order, ordered=True)
                mouse_data = mouse_data.sort_index()
            
            if mouse == "JAL7":
                # Set the index as a Categorical type with the specified order
                mouse_data.index = pd.Categorical(mouse_data.index, categories=JAL7_session_order, ordered=True)
                mouse_data = mouse_data.sort_index()

            # Plot each column (data series) for the current mouse
            for column in mouse_data.columns:
                axs[row_idx, col_idx].plot(mouse_data[column], marker='o', label=column, linewidth=3, markersize=8)
                axs[row_idx, col_idx].set_title(f'Mouse: {mouse}')
                
            axs[row_idx, col_idx].legend(loc="upper right")

            col_idx += 1  # Move to the next column for the next mouse

        # Adjust layout to prevent overlap
        plt.tight_layout()
        plt.show()

# Example usage
plot_mice_on_separate_axes(con_dict, conditions, mice_names)


# Sessions to drop

In [213]:
tinny_barrier = ["JAL7_30th_April", "JAL8_21st_May", "JAL8_3rd_May"]

# Box plot per condition

In [36]:
con_dict

{'shelter_only':                     frames  mouse_x_position  mouse_y_position      hdir  \
 JAL7_sesh9_16apr -1.253504         -0.124939          0.020482 -0.191449   
 
                        hsa  h_preflipbar_a  h_postflipbar_a  homing_id  \
 JAL7_sesh9_16apr  0.342075       -0.695345         0.459967  -2.187501   
 
                   post_flip_index  pre_flip_index  velocity    random  
 JAL7_sesh9_16apr         0.096923         0.12872 -0.035122 -0.279529  ,
 'barrier_pre_flip':                     frames  mouse_x_position  mouse_y_position     hdir  \
 JAL7_sesh9_16apr -1.561563         -0.771006         -1.219018 -0.53034   
 
                        hsa  h_preflipbar_a  h_postflipbar_a  homing_id  \
 JAL7_sesh9_16apr -0.453665        0.388471         0.131195  -3.559021   
 
                   post_flip_index  pre_flip_index  velocity    random  
 JAL7_sesh9_16apr        -0.508191        -0.24408 -1.782771 -0.360644  ,
 'barrier_post_flip':                     frames  mouse_

In [37]:
plt.close()

In [38]:
#test = con_dict["shelter_only"].drop(index=["JAL7_30th_April", "JAL8_21st_May"])
df = con_dict["barrier_post_flip"]
df = df.drop(columns=["frames", "homing_id", "mouse_x_position", "random", "mouse_y_position", "h_postflipbar_a", "h_preflipbar_a"])
#df[df < 0] = 0  # Clip negative values to zero
df_melted = df.reset_index().melt(id_vars='index', var_name='Feature', value_name='Value')
# Rename the 'index' column to 'Session' or whatever you want to call the row labels
df_melted = df_melted.rename(columns={'index': 'Session'})

#----------------------- order

# Calculate the median value for each feature
feature_median = df_melted.groupby('Feature')['Value'].median()

# Sort the features by the median values in descending order
sorted_features = feature_median.sort_values(ascending=False).index

# Reorder the 'Feature' column based on the sorted features
df_melted['Feature'] = pd.Categorical(df_melted['Feature'], categories=sorted_features, ordered=True)

# Now plot the boxplot with the features in the specified order
sns.boxplot(data=df_melted, x='Value', y='Feature', color='skyblue')

# Set plot labels and title
plt.title("Post barrier flip", fontsize=24)
plt.xlabel("R2 Scores", fontsize=16)
plt.ylabel("")
plt.tight_layout()
plt.show()# Now plot the boxplot with the features in the specified order
sns.boxplot(data=df_melted, x='Value', y='Feature', palette='pastel')

C:\Users\laurence\AppData\Local\Temp\ipykernel_41548\802009929.py:29: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_melted, x='Value', y='Feature', palette='pastel')


<Axes: xlabel='Value', ylabel='Feature'>

In [ ]:
test = con_dict["barrier_pre_flip"].drop(index=tinny_barrier)
df = test
df = df.drop(columns=["frames", "homing_id", "mouse_x_position", "random", "mouse_y_position", "h_postflipbar_a", "h_preflipbar_a"])
df[df < 0] = 0  # Clip negative values to zero
df_melted = df.reset_index().melt(id_vars='index', var_name='Feature', value_name='Value')
# Rename the 'index' column to 'Session' or whatever you want to call the row labels
df_melted = df_melted.rename(columns={'index': 'Session'})

### order by median

# Calculate the median value for each feature
feature_median = df_melted.groupby('Feature')['Value'].median()

# Sort the features by the median values in descending order
sorted_features = feature_median.sort_values(ascending=False).index

# Reorder the 'Feature' column based on the sorted features
df_melted['Feature'] = pd.Categorical(df_melted['Feature'], categories=sorted_features, ordered=True)

# Now plot the boxplot with the features in the specified order
sns.boxplot(data=df_melted, x='Value', y='Feature', color='skyblue')

# Set plot labels and title
plt.title("Pre flip condition", fontsize=24)
plt.xlabel("R2 Scores", fontsize=16)
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
test = con_dict["barrier_post_flip"].drop(index=tinny_barrier)
df = test
df = df.drop(columns=["frames", "homing_id", "mouse_x_position", "random", "mouse_y_position", "h_postflipbar_a", "h_preflipbar_a"])
df[df < 0] = 0  # Clip negative values to zero
df_melted = df.reset_index().melt(id_vars='index', var_name='Feature', value_name='Value')
# Rename the 'index' column to 'Session' or whatever you want to call the row labels
df_melted = df_melted.rename(columns={'index': 'Session'})

# --- sort

feature_median = df_melted.groupby('Feature')['Value'].median()

# Sort the features by the median values in descending order
sorted_features = feature_median.sort_values(ascending=False).index

# Reorder the 'Feature' column based on the sorted features
df_melted['Feature'] = pd.Categorical(df_melted['Feature'], categories=sorted_features, ordered=True)

# Now plot the boxplot with the features in the specified order
sns.boxplot(data=df_melted, x='Value', y='Feature', color='skyblue')

# Set plot labels and title
plt.title("Post flip condition", fontsize=24)
plt.xlabel("R2 Scores", fontsize=16)
plt.ylabel("")
plt.tight_layout()
plt.show()


###